## PACOTES ##

In [20]:
import pandas as pd
import patsy
import statsmodels.api as sm

# ETL #


### Primeiro Contato ###

In [21]:
# Ler o arquivo CSV
df = pd.read_csv("thoracic_surgery_data.csv")

# Diagnóstico inicial
linhas = df.shape[0]
variaveis = df.shape[1]
dados_faltantes = df.isnull().sum().sum()
linhas_duplicadas = df.duplicated().sum()

# Criar tabela resumo
diagnostico = pd.DataFrame({
    "Informação": [
        "Número de linhas",
        "Número de variáveis",
        "Total de dados faltantes",
        "Número de linhas duplicadas"
    ],
    "Resultado": [
        linhas,
        variaveis,
        dados_faltantes,
        linhas_duplicadas
    ]
})

diagnostico

,Informação,Resultado
0,Número de linhas,470
1,Número de variáveis,17
2,Total de dados faltantes,0
3,Número de linhas duplicadas,0


### TROCAR NOME DAS VARIÁVEIS PREDITORAS ###

In [22]:

df = df.rename(columns={
    "DGN": "diagnostico",
    "PRE4": "capacidade_vital_forcada",
    "PRE5": "volume_expiratorio_forcado_1s",
    "PRE6": "estado_desempenho_zubrod",
    "PRE7": "dor_antes_cirurgia",
    "PRE8": "hemoptise_antes_cirurgia",
    "PRE9": "dispneia_antes_cirurgia",
    "PRE10": "tosse_antes_cirurgia",
    "PRE11": "fraqueza_antes_cirurgia",
    "PRE14": "tamanho_tumor_tnm",
    "PRE17": "diabetes_mellitus_tipo_2",
    "PRE19": "infarto_miocardio_ate_6_meses",
    "PRE25": "doenca_arterial_periferica",
    "PRE30": "tabagismo",
    "PRE32": "asma",
    "AGE": "idade",
    "Risk1Y": "obito_1_ano",
    "Risk1Yr": "obito_1_ano"
})

# Conferir os novos nomes das colunas
df.columns

Index(['diagnostico', 'capacidade_vital_forcada',
       'volume_expiratorio_forcado_1s', 'estado_desempenho_zubrod',
       'dor_antes_cirurgia', 'hemoptise_antes_cirurgia',
       'dispneia_antes_cirurgia', 'tosse_antes_cirurgia',
       'fraqueza_antes_cirurgia', 'tamanho_tumor_tnm',
       'diabetes_mellitus_tipo_2', 'infarto_miocardio_ate_6_meses',
       'doenca_arterial_periferica', 'tabagismo', 'asma', 'idade',
       'obito_1_ano'],
      dtype='object')

In [23]:
df.to_csv("thoracic_surgery_variaveis_pt.csv", index=False, encoding="utf-8-sig")

print("Arquivo salvo como: thoracic_surgery_variaveis_pt.csv")

Arquivo salvo como: thoracic_surgery_variaveis_pt.csv


### Avaliação e alteração das variáveis preditoras CATEGORIZADAS "DGN" ### 

In [24]:

# Lendo a base
dados = pd.read_csv("thoracic_surgery_variaveis_pt.csv")

# Verificando os nomes das colunas
print(dados.columns)

# Quantos tipos diferentes de DGN existem
qtd_dgn = dados["diagnostico"].nunique()

print("Quantidade de tipos diferentes de DGN:", qtd_dgn)

# Quais são os tipos de DGN
print(dados["diagnostico"].unique())

# Frequência de cada tipo de DGN
print(dados["diagnostico"].value_counts())

Index(['diagnostico', 'capacidade_vital_forcada',
       'volume_expiratorio_forcado_1s', 'estado_desempenho_zubrod',
       'dor_antes_cirurgia', 'hemoptise_antes_cirurgia',
       'dispneia_antes_cirurgia', 'tosse_antes_cirurgia',
       'fraqueza_antes_cirurgia', 'tamanho_tumor_tnm',
       'diabetes_mellitus_tipo_2', 'infarto_miocardio_ate_6_meses',
       'doenca_arterial_periferica', 'tabagismo', 'asma', 'idade',
       'obito_1_ano'],
      dtype='object')
Quantidade de tipos diferentes de DGN: 7
['DGN2' 'DGN3' 'DGN4' 'DGN8' 'DGN5' 'DGN6' 'DGN1']
diagnostico
DGN3    349
DGN2     52
DGN4     47
DGN5     15
DGN6      4
DGN8      2
DGN1      1
Name: count, dtype: int64


### Avaliação e alteração das variáveis preditoras CATEGORIZADAS "DGN" ### 

In [25]:
# Lendo a base
dados = pd.read_csv("thoracic_surgery_variaveis_pt.csv")

# Verificando os nomes das colunas
print(dados.columns)

# Quantos tipos diferentes de DGN existem
qtd_dgn = dados["estado_desempenho_zubrod"].nunique()

print("Quantidade de tipos diferentes de estados do desempenho de zubrod:", qtd_dgn)

# Quais são os tipos de DGN
print(dados["estado_desempenho_zubrod"].unique())

# Frequência de cada tipo de DGN
print(dados["estado_desempenho_zubrod"].value_counts())

Index(['diagnostico', 'capacidade_vital_forcada',
       'volume_expiratorio_forcado_1s', 'estado_desempenho_zubrod',
       'dor_antes_cirurgia', 'hemoptise_antes_cirurgia',
       'dispneia_antes_cirurgia', 'tosse_antes_cirurgia',
       'fraqueza_antes_cirurgia', 'tamanho_tumor_tnm',
       'diabetes_mellitus_tipo_2', 'infarto_miocardio_ate_6_meses',
       'doenca_arterial_periferica', 'tabagismo', 'asma', 'idade',
       'obito_1_ano'],
      dtype='object')
Quantidade de tipos diferentes de estados do desempenho de zubrod: 3
['PRZ1' 'PRZ0' 'PRZ2']
estado_desempenho_zubrod
PRZ1    313
PRZ0    130
PRZ2     27
Name: count, dtype: int64


VARIÁVEL TARGET

In [26]:
import pandas as pd

target = "obito_1_ano"

freq_abs = dados[target].value_counts().sort_index()
freq_rel = dados[target].value_counts(normalize=True).sort_index() * 100

tabela_target = pd.DataFrame({
    "classe": freq_abs.index,
    "frequencia_absoluta": freq_abs.values,
    "frequencia_relativa_percentual": freq_rel.round(2).values
})

tabela_target

,classe,frequencia_absoluta,frequencia_relativa_percentual
0,F,400,85.11
1,T,70,14.89


## Transformações totais ##

In [27]:
# ============================================================
# 1. Ler a base
# ============================================================

dados = pd.read_csv("thoracic_surgery_variaveis_pt.csv")

# Remover espaços extras em colunas de texto
for col in dados.select_dtypes(include="object").columns:
    dados[col] = dados[col].astype(str).str.strip()

In [28]:
# ============================================================
# 2. Transformar T/F em 1/0
# ============================================================

variaveis_binarias = [
    "dor_antes_cirurgia",
    "hemoptise_antes_cirurgia",
    "dispneia_antes_cirurgia",
    "tosse_antes_cirurgia",
    "fraqueza_antes_cirurgia",
    "diabetes_mellitus_tipo_2",
    "infarto_miocardio_ate_6_meses",
    "doenca_arterial_periferica",
    "tabagismo",
    "asma",
    "obito_1_ano"
]

mapa_tf = {
    "T": 1,
    "TRUE": 1,
    "1": 1,
    "F": 0,
    "FALSE": 0,
    "0": 0
}

for var in variaveis_binarias:
    dados[var] = (
        dados[var]
        .astype(str)
        .str.strip()
        .str.upper()
        .map(mapa_tf)
    )

# Conferir se alguma conversão deu problema
print(dados[variaveis_binarias].isna().sum())

# Converter para inteiro
for var in variaveis_binarias:
    dados[var] = dados[var].astype(int)

dor_antes_cirurgia               0
hemoptise_antes_cirurgia         0
dispneia_antes_cirurgia          0
tosse_antes_cirurgia             0
fraqueza_antes_cirurgia          0
diabetes_mellitus_tipo_2         0
infarto_miocardio_ate_6_meses    0
doenca_arterial_periferica       0
tabagismo                        0
asma                             0
obito_1_ano                      0
dtype: int64


In [29]:
# ============================================================
# 3. Fórmula com referências escolhidas
# ============================================================

formula = """
obito_1_ano ~
C(diagnostico, Treatment(reference='DGN3'))
+ C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))
+ C(tamanho_tumor_tnm, Treatment(reference='OC11'))
+ capacidade_vital_forcada
+ volume_expiratorio_forcado_1s
+ dor_antes_cirurgia
+ hemoptise_antes_cirurgia
+ dispneia_antes_cirurgia
+ tosse_antes_cirurgia
+ fraqueza_antes_cirurgia
+ diabetes_mellitus_tipo_2
+ infarto_miocardio_ate_6_meses
+ doenca_arterial_periferica
+ tabagismo
+ asma
+ idade
"""

In [30]:
# ============================================================
# 4. Criar matriz do modelo com dummies
# ============================================================

y, X = patsy.dmatrices(
    formula,
    data=dados,
    return_type="dataframe"
)

# Juntar variável resposta com variáveis explicativas transformadas
dados_modelo = pd.concat([y, X], axis=1)

# Opcional: renomear Intercept para constante
dados_modelo = dados_modelo.rename(columns={"Intercept": "constante"})

print(dados_modelo.head())
print(dados_modelo.columns)

   obito_1_ano  constante  \
0          0.0        1.0   
1          0.0        1.0   
2          0.0        1.0   
3          0.0        1.0   
4          1.0        1.0   

   C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]  \
0                                                0.0     
1                                                0.0     
2                                                0.0     
3                                                0.0     
4                                                0.0     

   C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]  \
0                                                1.0     
1                                                0.0     
2                                                0.0     
3                                                0.0     
4                                                0.0     

   C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]  \
0                                                0.0     
1          

In [33]:
# ============================================================
# 5. Salvar a nova base
# ============================================================

dados_modelo.to_csv(
    "thoracic_surgery_base_modelo_statsmodels.csv",
    index=False
)

print("Base transformada salva como: thoracic_surgery_base_modelo_statsmodels.csv")

Base transformada salva como: thoracic_surgery_base_modelo_statsmodels.csv


## Verificação se transformação ocorreu bem##

In [34]:

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

# ============================================================
# 2. Informações gerais da base
# ============================================================

print("=" * 80)
print("INFORMAÇÕES GERAIS DA BASE")
print("=" * 80)

print(f"Número de linhas: {dados_modelo.shape[0]}")
print(f"Número de colunas: {dados_modelo.shape[1]}")
print(f"Total de dados faltantes: {dados_modelo.isna().sum().sum()}")
print(f"Número de linhas duplicadas: {dados_modelo.duplicated().sum()}")

print("\nColunas da base:")
for coluna in dados_modelo.columns:
    print("-", coluna)

# ============================================================
# 3. Resumo: quantidade de valores diferentes e quais são
# ============================================================

resumo = []

for coluna in dados_modelo.columns:
    valores_unicos = dados_modelo[coluna].dropna().unique()
    valores_unicos_ordenados = sorted(valores_unicos, key=lambda x: str(x))
    quantidade = dados_modelo[coluna].nunique(dropna=True)

    conjunto_valores = set(valores_unicos_ordenados)

    if coluna == "obito_1_ano":
        tipo_detectado = "Variável resposta binária"
    elif quantidade == 1:
        tipo_detectado = "Constante"
    elif conjunto_valores.issubset({0, 1, 0.0, 1.0}):
        tipo_detectado = "Binária / Dummy"
    else:
        tipo_detectado = "Numérica"

    resumo.append({
        "variavel": coluna,
        "tipo_detectado": tipo_detectado,
        "quantidade_valores_diferentes": quantidade,
        "valores_diferentes": valores_unicos_ordenados
    })

resumo_variaveis = pd.DataFrame(resumo)

print("\n" + "=" * 80)
print("RESUMO DAS VARIÁVEIS")
print("=" * 80)

display(resumo_variaveis)

# ============================================================
# 4. Visualizar frequência das variáveis binárias e dummies
# ============================================================

print("\n" + "=" * 80)
print("FREQUÊNCIA DAS VARIÁVEIS BINÁRIAS / DUMMIES")
print("=" * 80)

for coluna in dados_modelo.columns:
    valores = set(dados_modelo[coluna].dropna().unique())

    if valores.issubset({0, 1, 0.0, 1.0}):
        print("\nVariável:", coluna)
        print(dados_modelo[coluna].value_counts().sort_index())

# ============================================================
# 5. Visualizar estatísticas das variáveis numéricas
# ============================================================

variaveis_numericas = []

for coluna in dados_modelo.columns:
    valores = set(dados_modelo[coluna].dropna().unique())

    if not valores.issubset({0, 1, 0.0, 1.0}):
        variaveis_numericas.append(coluna)

print("\n" + "=" * 80)
print("ESTATÍSTICAS DAS VARIÁVEIS NUMÉRICAS")
print("=" * 80)

display(dados_modelo[variaveis_numericas].describe().T)

# ============================================================
# 6. Distribuição da variável resposta
# ============================================================

print("\n" + "=" * 80)
print("DISTRIBUIÇÃO DA VARIÁVEL RESPOSTA")
print("=" * 80)

print("\nFrequência absoluta:")
print(dados_modelo["obito_1_ano"].value_counts().sort_index())

print("\nFrequência relativa (%):")
print((dados_modelo["obito_1_ano"].value_counts(normalize=True).sort_index() * 100).round(2))

INFORMAÇÕES GERAIS DA BASE
Número de linhas: 470
Número de colunas: 26
Total de dados faltantes: 0
Número de linhas duplicadas: 0

Colunas da base:
- obito_1_ano
- constante
- C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]
- C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]
- C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]
- C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]
- C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]
- C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]
- C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0]
- C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]
- C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC12]
- C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC13]
- C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC14]
- capacidade_vital_forcada
- volume_expiratorio_forcado_1s
- dor_antes_cirurgia
- hemoptise_antes_cirurgia
- dispneia_antes_cirurgia
- tosse_antes_cirurgia
- fraqueza_antes_cirurgia
- diab

,variavel,tipo_detectado,quantidade_valores_diferentes,valores_diferentes
0,obito_1_ano,Variável resposta binária,2,"[0.0, 1.0]"
1,constante,Constante,1,[1.0]
2,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]",Binária / Dummy,2,"[0.0, 1.0]"
3,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]",Binária / Dummy,2,"[0.0, 1.0]"
4,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]",Binária / Dummy,2,"[0.0, 1.0]"
5,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]",Binária / Dummy,2,"[0.0, 1.0]"
6,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]",Binária / Dummy,2,"[0.0, 1.0]"
7,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]",Binária / Dummy,2,"[0.0, 1.0]"
8,"C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0]",Binária / Dummy,2,"[0.0, 1.0]"
9,"C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]",Binária / Dummy,2,"[0.0, 1.0]"



FREQUÊNCIA DAS VARIÁVEIS BINÁRIAS / DUMMIES

Variável: obito_1_ano
obito_1_ano
0.0    400
1.0     70
Name: count, dtype: int64

Variável: constante
constante
1.0    470
Name: count, dtype: int64

Variável: C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]
C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]
0.0    469
1.0      1
Name: count, dtype: int64

Variável: C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]
C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]
0.0    418
1.0     52
Name: count, dtype: int64

Variável: C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]
C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]
0.0    423
1.0     47
Name: count, dtype: int64

Variável: C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]
C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]
0.0    455
1.0     15
Name: count, dtype: int64

Variável: C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]
C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]
0.0    466
1.0      4
Name: count, dt

,count,mean,std,min,25%,50%,75%,max
capacidade_vital_forcada,470.0,3.281638,0.871395,1.44,2.60,3.16,3.8075,6.3
volume_expiratorio_forcado_1s,470.0,4.568702,11.767857,0.96,1.96,2.40,3.0800,86.3
idade,470.0,62.534043,8.706902,21.00,57.00,62.00,69.0000,87.0



DISTRIBUIÇÃO DA VARIÁVEL RESPOSTA

Frequência absoluta:
obito_1_ano
0.0    400
1.0     70
Name: count, dtype: int64

Frequência relativa (%):
obito_1_ano
0.0    85.11
1.0    14.89
Name: proportion, dtype: float64
